# Week 6 Lab — A Grounded, Stateful Agent

**Duration: 2 hours.** Everything you need is in this folder: setup, environment, the corpus, the notebook, and the API app. Week 5 is *not* a prerequisite for the plumbing — only for the ideas.

**Goal:** build a support agent for the fictional retailer **Contoso Outdoor Gear** that

- answers **only** from the company handbook (local Chroma + hybrid search you implement yourself),
- **self-corrects** a bad retrieval (grade → rewrite → retry, as an explicit LangGraph),
- **remembers** the conversation (checkpointing),
- and **pauses for a human** before issuing a refund (interrupts) — over HTTP, in Part 6.

| Time | Part |
|---|---|
| 0:00–0:10 | **0.** Setup & environment check |
| 0:10–0:30 | **1.** Ingest: corpus → chunks → vectors, and hybrid search ✍️ |
| 0:30–0:50 | **2.** Agentic RAG with `create_agent` |
| 0:50–1:20 | **3.** The explicit graph: grade & self-correct ⭐ |
| 1:20–1:33 | **4.** Memory via checkpointing |
| 1:33–1:45 | **5.** Human-in-the-loop refunds |
| 1:45–1:55 | **6.** Serve the whole workflow behind FastAPI |
| 1:55–2:00 | Wrap-up & stretch goals |

**Rules.** Type the ✍️ cells. `solutions.ipynb` is for after a real attempt. Each ✅ **CHECKPOINT** is a sign-off point.

**Budget:** ~45 model requests. The Gemini free tier is metered per *minute* as well as per day; a `429` means wait a moment, not that your day is over. Out of quota entirely? Pair up: one `.env`, one laptop driving.

## Part 0 — Setup & environment (0:00–0:10)

Full instructions, including how to create the free key, are in this folder's [README.md](README.md); the complete written walkthrough — every code component, explained — is in [labsheet.md](labsheet.md). The short version, run **from this folder** (`week-6/lab/`):

```bash
python3 -m venv .venv
source .venv/bin/activate                 # Windows: .venv\Scripts\activate
pip install -r requirements.txt
python -m ipykernel install --user --name agentic-w6 --display-name "Agentic AI (week 6)"
cp .env.example .env                      # then paste your Gemini API key into GOOGLE_API_KEY
jupyter lab lab.ipynb
```

**Pick the kernel before running anything:** top-right of this notebook (or *Kernel ▸ Change Kernel…*), choose **"Agentic AI (week 6)"**. A notebook left on the default *Python 3* kernel runs a different interpreter than your `.venv`, and every import fails even though the install worked. The next cell checks this for you.

Week 6 needs **two** models: a chat model *and* an embedding model — both free on the Gemini API, both already in `.env.example`.

In [1]:
# ── Kernel check — run this FIRST ──────────────────────────────────────────
# Standard library only, so it works even when nothing else is installed.
import importlib.util
import sys
from pathlib import Path

LAB_DIR = Path.cwd()
LAB_VENV = LAB_DIR / ".venv"
in_a_venv = sys.prefix != sys.base_prefix
on_lab_venv = in_a_venv and Path(sys.prefix).resolve() == LAB_VENV.resolve()

print("this kernel runs :", sys.executable)
print("python version   :", sys.version.split()[0])
print("environment      :", sys.prefix)
print("on this lab .venv:", "yes" if on_lab_venv else "no")

REQUIRED = ["dotenv", "langchain", "langchain_google_genai", "langgraph",
            "langchain_chroma", "chromadb", "langchain_community", "rank_bm25",
            "fastapi", "uvicorn", "requests"]
missing = [m for m in REQUIRED if importlib.util.find_spec(m) is None]
print("missing packages :", ", ".join(missing) if missing else "none")

if missing:
    print(rf"""
FIX — you are almost certainly on the wrong kernel, not missing an install.

 1. Kernel menu -> Change Kernel... -> "Agentic AI (week 6)", then re-run this cell.

 2. No such kernel in the list? Register it. In a terminal, from
    {LAB_DIR}
        source .venv/bin/activate          # Windows: .venv\Scripts\activate
        pip install -r requirements.txt
        python -m ipykernel install --user --name agentic-w6 --display-name "Agentic AI (week 6)"
    then reload this browser tab and do step 1.

 3. Or launch Jupyter FROM the venv, so its default kernel is already correct:
        source .venv/bin/activate
        python -m jupyterlab lab.ipynb

 4. Last resort — install into whatever kernel you are on, then restart the kernel:
        %pip install -r requirements.txt
""")
    raise SystemExit("Wrong kernel or missing packages — see the fix printed above.")

print("\nReady. Continue to the next cell.")

this kernel runs : D:\SE3090_Lab06_Agentic_AI_Part_2\SE3090_Lab06_Agentic_AI_Part_2\.venv\Scripts\python.exe
python version   : 3.13.5
environment      : D:\SE3090_Lab06_Agentic_AI_Part_2\SE3090_Lab06_Agentic_AI_Part_2\.venv
on this lab .venv: yes
missing packages : none

Ready. Continue to the next cell.


In [2]:
# Environment check 1/2 — configuration only, no model call
import os
from pathlib import Path

from dotenv import load_dotenv

LAB_DIR = Path.cwd()
load_dotenv(LAB_DIR / ".env")
load_dotenv(override=True)
API_KEY = os.getenv("GOOGLE_API_KEY", "")
CHAT_MODEL = os.getenv("CHAT_MODEL", "gemini-3.5-flash")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "gemini-embedding-001")
CHROMA_DIR = str(LAB_DIR / "chroma_db")
COLLECTION = os.getenv("VECTOR_COLLECTION", "contoso-handbook")

assert API_KEY and "XXXX" not in API_KEY, "GOOGLE_API_KEY missing: copy .env.example -> .env"
print(f"chat      : {CHAT_MODEL}")
print(f"embeddings: {EMBEDDING_MODEL}")
print(f"chroma    : {CHROMA_DIR}")
print(f"corpus    : {[p.name for p in sorted((LAB_DIR / 'data').glob('*.md'))]}")

chat      : gemini-3.5-flash-lite
embeddings: gemini-embedding-001
chroma    : D:\SE3090_Lab06_Agentic_AI_Part_2\SE3090_Lab06_Agentic_AI_Part_2\chroma_db
corpus    : ['product-faq.md', 'returns-policy.md', 'warranty-guide.md']


In [3]:
# Environment check 2/2 — one chat call + one embedding call (2 requests)
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

llm = ChatGoogleGenerativeAI(model=CHAT_MODEL, google_api_key=API_KEY,
                             temperature=0, timeout=60,
                             max_retries=3)   # free tier: back off and retry on 429

embeddings = GoogleGenerativeAIEmbeddings(model=EMBEDDING_MODEL, google_api_key=API_KEY)

print("chat  ->", llm.invoke("Reply with exactly: OK").content)
print("embed ->", f"{len(embeddings.embed_query('hello agents'))}-dimensional vector")

D:\SE3090_Lab06_Agentic_AI_Part_2\SE3090_Lab06_Agentic_AI_Part_2\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


chat  -> [{'type': 'text', 'text': 'OK', 'extras': {'signature': 'El4KXAERTTIPCfeVgfy3vsq6w0JkNH7oEmLZ5cswx2XIX4cGlQHjFpdmxVCEwWnGGjRqHSD5GGYU/AyDaRmrSABg1O1o4s2VYT3YJeF2jhPMAP3hFH+OF6Lod/h76+ug'}}]
embed -> 3072-dimensional vector


## Part 1 — Ingest and hybrid search (0:10–0:30)

Skim `data/` first — three markdown files, a five-minute read. **Note the trap built into the corpus: tents have a 14-day return window but a 3-year pole warranty.** It will decide several answers today.

The offline half of RAG is four steps: **load → chunk → embed → store**. The next cell is all four.

In [4]:
# Ingest (given). Idempotent: re-run it any time; it replaces the collection.
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

vector_store = Chroma(
    collection_name=COLLECTION,
    embedding_function=embeddings,
    persist_directory=CHROMA_DIR,      # ← your "vector database" is this folder
)


def ingest() -> int:
    docs = [                                                   # 1. load
        Document(page_content=p.read_text(encoding="utf-8"), metadata={"source": p.stem})
        for p in sorted((LAB_DIR / "data").glob("*.md"))
    ]
    splitter = RecursiveCharacterTextSplitter(                 # 2. chunk
        chunk_size=800,          # ~200 tokens: small enough to be precise
        chunk_overlap=120,       # so facts straddling a cut survive
        separators=["\n## ", "\n\n", "\n", " "],   # prefer heading/paragraph cuts
    )
    chunks = splitter.split_documents(docs)

    existing = vector_store.get()["ids"]
    if existing:
        vector_store.delete(ids=existing)
    vector_store.add_documents(chunks)                         # 3+4. embed & store
    return len(chunks)


print(f"indexed {ingest()} chunks into {CHROMA_DIR}")

indexed 8 chunks into D:\SE3090_Lab06_Agentic_AI_Part_2\SE3090_Lab06_Agentic_AI_Part_2\chroma_db


In [5]:
# Look inside: a "vector DB" is rows of (text, vector, metadata). Nothing more.
data = vector_store.get(include=["embeddings", "documents", "metadatas"])
vec = data["embeddings"][0]
print(f"{len(data['ids'])} chunks on disk\n")
print(f"chunk 0 [{data['metadatas'][0]['source']}]:")
print(" ", data["documents"][0][:200].replace("\n", " "), "…")
print(f"  vector: dim={len(vec)}, first 6 = {[round(float(x), 4) for x in vec[:6]]}")

8 chunks on disk

chunk 0 [product-faq]:
  # Contoso Outdoor Gear — Product FAQ  ## SummitPro X2 tent  - 2-person, 3-season tent. **Not rated for winter/4-season use.** - Weight: **2.4 kg** packed; floor area 2.9 m². - Waterproofing: **3000 mm …
  vector: dim=3072, first 6 = [0.0053, 0.0063, 0.0146, -0.0664, 0.0161, 0.0087]


In [6]:
# Pure VECTOR search: meaning, not words. Note the query shares no keywords with the hit.
for d in vector_store.similarity_search("can I get my money back", k=2):
    print(f"[{d.metadata['source']}] {d.page_content[:110].replace(chr(10), ' ')}…\n")

[returns-policy] ## Refund processing  Refunds are issued to the **original payment method** and typically arrive within **5–10…

[returns-policy] # Contoso Outdoor Gear — Returns & Refunds Policy (2026 edition)  ## Standard return window  Most items may be…



Vector search has two well-known blind spots: it blurs **negation** ("eligible" vs "NOT eligible" embed almost identically) and it is weak on **exact identifiers** (part codes, order numbers) where plain keyword matching wins.

That is why production retrieval is **hybrid**: run BM25 (keywords) and vector search in parallel, then fuse the two ranked lists with **Reciprocal Rank Fusion** — each list contributes `1/(60 + rank)` per document, scores are summed, and the top `k` win. Because RRF is *rank*-based, BM25 scores and cosine similarities never need calibrating against each other.

Now write it. It is genuinely ~15 lines.

In [7]:
# ✍️ TODO 1 — implement hybrid search with Reciprocal Rank Fusion.
from langchain_community.retrievers import BM25Retriever

# The keyword half: BM25 over every chunk in the store (fine at course scale).
_rows = vector_store.get()
bm25 = BM25Retriever.from_documents(
    [Document(page_content=t, metadata=m or {}) for t, m in zip(_rows["documents"], _rows["metadatas"])]
)
bm25.k = 8


def hybrid_search(query: str, k: int = 3) -> list[Document]:
    """BM25 + vector search, fused with RRF."""
    vector_hits = vector_store.similarity_search(query, k=8)
    keyword_hits = bm25.invoke(query)

    # TODO a) scores: dict[str, float] and by_key: dict[str, Document]
    scores: dict[str, float] = {}
    by_key: dict[str, Document] = {}

    # TODO b) for each of the two hit lists, for (rank, doc) in enumerate(hits):
    #            key = doc.page_content[:80]        # cheap identity across both lists
    #            by_key[key] = doc
    #            scores[key] = scores.get(key, 0.0) + 1.0 / (60 + rank)
    for hits in [vector_hits, keyword_hits]:
        for rank, doc in enumerate(hits):
            key = doc.page_content[:80]
            by_key[key] = doc
            scores[key] = scores.get(key, 0.0) + 1.0 / (60 + rank)

    # TODO c) return the k documents with the highest fused score
    ranked_keys = sorted(scores, key=scores.get, reverse=True)
    return [by_key[key] for key in ranked_keys[:k]]


def format_docs(docs: list[Document]) -> str:
    """Render results the way the model will see them: [source] then text."""
    return "\n\n".join(
        f"[{d.metadata.get('source', 'unknown')}]\n{d.page_content.strip()}" for d in docs
    ) or "No results."


def search_formatted(query: str, k: int = 3) -> str:
    return format_docs(hybrid_search(query, k))


print(search_formatted("restocking fee opened electronics", k=2))

C:\Users\USER\AppData\Local\Temp\ipykernel_37300\352850446.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


[returns-policy]
## Refund processing

Refunds are issued to the **original payment method** and typically arrive within
**5–10 business days** of us receiving the item. We will email you at each step.

## Return shipping

- Orders over **$75**: return shipping is **free** (prepaid label emailed to you).
- Orders $75 and under: a flat **$8.50** label fee is deducted from the refund.

## Exchanges

Exchanges for a different size or color are **free** within the applicable return
window, subject to stock. Start an exchange from your order page.

## How to start a return

Sign in → Orders → select the item → "Start a return". You will receive a QR code
for any courier drop-off point. No printer needed.

[returns-policy]
# Contoso Outdoor Gear — Returns & Refunds Policy (2026 edition)

## Standard return window

Most items may be returned within **30 days of delivery** for a full refund,
provided they are unused, in original packaging, and with tags attached.

## Category exceptions

- **T

In [8]:
# ✍️ EXERCISE 1.1 — three probes. Write one sentence of observation for each.
#   a) "can I get my money back"      — semantic hit, no shared keywords
#   b) "AquaPure cartridge litres"    — an exact-ish identifier: does BM25 rescue it?
#   c) "do you sell kayaks"           — NOT in the corpus. What comes back anyway?
for q in ["can I get my money back", "AquaPure cartridge litres", "do you sell kayaks"]:
    print("=" * 70, f"\nQUERY: {q}")
    for d in hybrid_search(q, k=2):
        print(f"  [{d.metadata['source']}] {d.page_content[:90].replace(chr(10), ' ')}…")

QUERY: can I get my money back
  [returns-policy] ## Refund processing  Refunds are issued to the **original payment method** and typically …
  [warranty-guide] ## Making a claim  1. Email **support@contoso-outdoor.example** with your order number. 2.…
QUERY: AquaPure cartridge litres
  [product-faq] ## AquaPure Trek filter  - Hollow-fibre filtration to **0.1 micron** — removes bacteria an…
  [warranty-guide] ## Coverage by product line  - **TrailBlazer backpacks:** **lifetime warranty** on stitchi…
QUERY: do you sell kayaks
  [warranty-guide] ## Making a claim  1. Email **support@contoso-outdoor.example** with your order number. 2.…
  [returns-policy] ## Refund processing  Refunds are issued to the **original payment method** and typically …


Probe (c) is the important one: **retrieval always returns something**. Relevance is a judgement your system has to make — which is exactly what you will build in Part 3.

✅ **CHECKPOINT 1** — Show the ingest output, your working `hybrid_search`, and the three probes.
*Demonstrator question: why did "can I get my money back" match the refund chunk with no shared keywords?*

## Part 2 — Agentic RAG with `create_agent` (0:30–0:50)

Classic RAG retrieves once, then answers. **Agentic RAG** hands retrieval to the agent as a *tool*, so it can reformulate your question, search several times, and decide when it has enough. The cost is more model calls; the benefit is everything below.

In [9]:
from langchain.agents import create_agent
from langchain_core.tools import tool


@tool
def search_handbook(query: str) -> str:
    """Search the Contoso Outdoor Gear company handbook (returns policy, warranty
    guide, product FAQ). Use a short, specific search query. Call this for EVERY
    factual question about products or policies; you may call it multiple times
    with different queries for multi-part questions."""
    return search_formatted(query, k=3)


SUPPORT_PROMPT = """You are the support agent for Contoso Outdoor Gear.

Rules:
- Answer ONLY from search_handbook results. Never rely on general knowledge about
  retail policies — Contoso's policies are unusual in places.
- Cite the source of every fact in square brackets, e.g. [returns-policy].
- Multi-part questions may need multiple searches with different queries.
- If, after searching, the answer is not in the results, say exactly that and suggest
  contacting support@contoso-outdoor.example. NEVER invent policy.
- Be concise and warm; lead with the answer, not the process."""

rag_agent = create_agent(llm, tools=[search_handbook], system_prompt=SUPPORT_PROMPT)


def ask_agent(question: str) -> None:
    print("=" * 72, f"\nQ: {question}")
    for chunk in rag_agent.stream(
        {"messages": [{"role": "user", "content": question}]}, stream_mode="values"
    ):
        chunk["messages"][-1].pretty_print()


ask_agent("What's the return window for a headlamp?")

Q: What's the return window for a headlamp?
================================ Human Message =================================

What's the return window for a headlamp?


D:\SE3090_Lab06_Agentic_AI_Part_2\SE3090_Lab06_Agentic_AI_Part_2\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


================================== Ai Message ==================================

[]
Tool Calls:
  search_handbook (call_652527)
 Call ID: call_652527
  Args:
    query: return window headlamp
================================= Tool Message =================================
Name: search_handbook

[returns-policy]
# Contoso Outdoor Gear — Returns & Refunds Policy (2026 edition)

## Standard return window

Most items may be returned within **30 days of delivery** for a full refund,
provided they are unused, in original packaging, and with tags attached.

## Category exceptions

- **Tents and shelters:** returnable within **14 days of delivery** only, due to
  seasonal demand. The tent must be unused; a tent that has been pitched outdoors
  is not eligible for return (see the Warranty Guide for defect coverage instead).
- **Electronics** (GPS units, headlamps, watches): 30-day window, but items with
  opened packaging incur a **15% restocking fee**.
- **Water filters:** returnable only if 

D:\SE3090_Lab06_Agentic_AI_Part_2\SE3090_Lab06_Agentic_AI_Part_2\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


================================== Ai Message ==================================

[{'type': 'text', 'text': 'The return window for a headlamp is **30 days of delivery**, but please note that items with opened packaging will incur a 15% restocking fee [returns-policy].', 'extras': {'signature': 'El4KXAERTTIP+DbN3Bn7uFLTRnbqXRIDC+jh9mXyVDXV/RCAAI/+G9O3aJjqeBW84FCBmhZqBY2rpmao2BH+FXBePKoOT9mdsL375AoXR+7QZWSvQop39qfQycPEGxd1'}}]


In [10]:
# THE TRAP — a correct answer needs TWO facts: 14-day tent returns (so: no) AND the
# 3-year pole warranty (so: yes, claim it). Did your agent search twice? Did it find both?
ask_agent("I bought a SummitPro tent 20 days ago and a pole snapped. Can I return it?")

Q: I bought a SummitPro tent 20 days ago and a pole snapped. Can I return it?
================================ Human Message =================================

I bought a SummitPro tent 20 days ago and a pole snapped. Can I return it?


D:\SE3090_Lab06_Agentic_AI_Part_2\SE3090_Lab06_Agentic_AI_Part_2\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


================================== Ai Message ==================================

[]
Tool Calls:
  search_handbook (call_647588)
 Call ID: call_647588
  Args:
    query: SummitPro tent return policy warranty broken pole
================================= Tool Message =================================
Name: search_handbook

[warranty-guide]
## Making a claim

1. Email **support@contoso-outdoor.example** with your order number.
2. Attach proof of purchase and 2–3 photos of the defect.
3. Our team assesses claims within **5 business days**.
4. Approved claims: we repair, or replace if repair is not feasible. **Shipping
   both ways is free** for warranty service.

## Common questions

- A snapped tent pole within 3 years of purchase **is covered** for SummitPro
  tents, even if the tent has been used — this is the most common approved claim.
- Warranty is non-transferable and requires the original proof of purchase.
- Commercial or rental use halves all warranty periods.

[warranty-guide]


D:\SE3090_Lab06_Agentic_AI_Part_2\SE3090_Lab06_Agentic_AI_Part_2\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


================================== Ai Message ==================================

[{'type': 'text', 'text': 'Yes, you are covered! While used tents cannot be returned for a standard refund (the return window for tents is 14 days and they must be unused) [returns-policy], a snapped tent pole falls under your **3-year SummitPro tent warranty** [warranty-guide]. \n\nTo make a warranty claim:\n1. Email **support@contoso-outdoor.example** with your order number [warranty-guide].\n2. Attach your proof of purchase and 2–3 photos of the broken pole [warranty-guide]. \n\nClaims are assessed within 5 business days, and if approved, Contoso will repair or replace the tent, with free shipping both ways [warranty-guide].', 'extras': {'signature': 'El4KXAERTTIPdA1n1p7jrXByeIKO7NJRKqdDzyCD9gvxDE/DRf59qYdHMeKZiC0RN1IkVFWL+45w3tDuT2CUA7XYBQoBlbpL61LTEq9C+zcV1tcN5fV04hM+86wxiO9n'}}]


In [11]:
# THE HONESTY TEST — kayaks are not in the handbook. The only correct answer says so.
ask_agent("Do you sell kayaks?")

Q: Do you sell kayaks?
================================ Human Message =================================

Do you sell kayaks?


D:\SE3090_Lab06_Agentic_AI_Part_2\SE3090_Lab06_Agentic_AI_Part_2\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


================================== Ai Message ==================================

[]
Tool Calls:
  search_handbook (call_823419)
 Call ID: call_823419
  Args:
    query: kayaks
================================= Tool Message =================================
Name: search_handbook

[warranty-guide]
## Making a claim

1. Email **support@contoso-outdoor.example** with your order number.
2. Attach proof of purchase and 2–3 photos of the defect.
3. Our team assesses claims within **5 business days**.
4. Approved claims: we repair, or replace if repair is not feasible. **Shipping
   both ways is free** for warranty service.

## Common questions

- A snapped tent pole within 3 years of purchase **is covered** for SummitPro
  tents, even if the tent has been used — this is the most common approved claim.
- Warranty is non-transferable and requires the original proof of purchase.
- Commercial or rental use halves all warranty periods.

[warranty-guide]
## Coverage by product line

- **TrailBlaze

D:\SE3090_Lab06_Agentic_AI_Part_2\SE3090_Lab06_Agentic_AI_Part_2\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


================================== Ai Message ==================================

[]
Tool Calls:
  search_handbook (call_828030)
 Call ID: call_828030
  Args:
    query: kayak
================================= Tool Message =================================
Name: search_handbook

[warranty-guide]
## Making a claim

1. Email **support@contoso-outdoor.example** with your order number.
2. Attach proof of purchase and 2–3 photos of the defect.
3. Our team assesses claims within **5 business days**.
4. Approved claims: we repair, or replace if repair is not feasible. **Shipping
   both ways is free** for warranty service.

## Common questions

- A snapped tent pole within 3 years of purchase **is covered** for SummitPro
  tents, even if the tent has been used — this is the most common approved claim.
- Warranty is non-transferable and requires the original proof of purchase.
- Commercial or rental use halves all warranty periods.

[returns-policy]
# Contoso Outdoor Gear — Returns & Refunds P

D:\SE3090_Lab06_Agentic_AI_Part_2\SE3090_Lab06_Agentic_AI_Part_2\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


================================== Ai Message ==================================

[{'type': 'text', 'text': 'Based on the information available in our handbook, kayaks are not mentioned among our product lines (which include TrailBlazer backpacks, SummitPro tents, AquaPure water filters, and electronics like NightHawk headlamps and GPS units). \n\nIf you have any further questions, please contact support@contoso-outdoor.example.', 'extras': {'signature': 'El4KXAERTTIPsp8D/LFY5xHMADnd+yvXy+usWfNBZgITBdqmaRv3IuI8islibIAWSu3PyZEXsOGeRH2//L5EeAdZuU+NPm7ynBDuHbfqDgigzWqcrnDfF1jczcSd8GvV'}}]


In [12]:
# ✍️ EXERCISE 2.1 — ask the trap question VAGUELY and compare the search queries the
# agent writes with what you typed. That gap is query rewriting: agentic benefit #1.
ask_agent("my tent thing broke, what can I do?")

Q: my tent thing broke, what can I do?
================================ Human Message =================================

my tent thing broke, what can I do?


D:\SE3090_Lab06_Agentic_AI_Part_2\SE3090_Lab06_Agentic_AI_Part_2\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


================================== Ai Message ==================================

[]
Tool Calls:
  search_handbook (call_914965)
 Call ID: call_914965
  Args:
    query: tent warranty returns repair
================================= Tool Message =================================
Name: search_handbook

[warranty-guide]
## Making a claim

1. Email **support@contoso-outdoor.example** with your order number.
2. Attach proof of purchase and 2–3 photos of the defect.
3. Our team assesses claims within **5 business days**.
4. Approved claims: we repair, or replace if repair is not feasible. **Shipping
   both ways is free** for warranty service.

## Common questions

- A snapped tent pole within 3 years of purchase **is covered** for SummitPro
  tents, even if the tent has been used — this is the most common approved claim.
- Warranty is non-transferable and requires the original proof of purchase.
- Commercial or rental use halves all warranty periods.

[warranty-guide]
# Contoso Outdoor Gea

D:\SE3090_Lab06_Agentic_AI_Part_2\SE3090_Lab06_Agentic_AI_Part_2\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


================================== Ai Message ==================================

[{'type': 'text', 'text': 'If your tent broke due to a manufacturing defect (such as a snapped tent pole within 3 years of purchase for SummitPro tents), it is covered by our warranty [warranty-guide]. \n\nTo make a warranty claim:\n1. Email **support@contoso-outdoor.example** with your order number [warranty-guide].\n2. Attach proof of purchase and 2–3 photos of the defect [warranty-guide]. \n\nOur team will assess your claim within 5 business days, and if approved, we will repair or replace it with free shipping both ways [warranty-guide]. Please note that warranties cover manufacturing defects rather than normal wear and tear or accidental damage [warranty-guide]. \n\nIf you are looking to return an unused tent rather than make a warranty claim, please note that tents and shelters must be returned within 14 days of delivery and must be unused (tents pitched outdoors are not eligible for return) [return

✅ **CHECKPOINT 2** — Show the trap-question trace with both facts found and cited, plus the kayak refusal.

> If your agent invented kayak policy, tighten `SUPPORT_PROMPT` and rerun. Grounding is a property you *engineer*, not one you hope for.

## Part 3 — The explicit graph: grade & self-correct (0:50–1:20) ⭐

`create_agent` cannot express *"grade the retrieval; if it is bad, rewrite the query and retry — at most twice, then answer honestly."* That is control flow, and control flow is what LangGraph is for. You now own the loop.

```
START → router ──"refund"──▶ human_gate → process_refund → END        (Parts 5)
          │
       "question"
          ▼
      retrieve → grade ──relevant──▶ answer → END
          ▲          │
          └─ rewrite ◀┘ irrelevant (max 2 retries, then answer honestly)
```

Five TODOs, in order. Run each cell as you go.

In [13]:
# ✍️ TODO 1 — the shared state.
# `messages` must APPEND across nodes; every other field replaces on write.
# Which annotation gives you appending? (You met it in the lecture: add_messages.)
from typing import Annotated, Literal, TypedDict

from langgraph.graph.message import add_messages


class State(TypedDict):
    messages: Annotated[list, add_messages]        # TODO: Annotated[list, ???]
    question: str
    search_query: str
    docs: str
    retries: int
    intent_: str         # router's classification ("question" | "refund")
    relevant_: bool      # grade's verdict
    approved_: bool      # human_gate's decision (Part 5)


MAX_RETRIES = 2

In [14]:
# The router is given — read it. Structured output turns an LLM into a typed function.
from pydantic import BaseModel, Field


class Intent(BaseModel):
    intent: Literal["question", "refund"] = Field(
        description="'refund' if the user is requesting a refund; else 'question'"
    )


def router(state: State) -> dict:
    result = llm.with_structured_output(Intent).invoke(
        f"Classify this customer message: {state['question']!r}"
    )
    print(f"[router] intent = {result.intent}")
    return {"search_query": state["question"], "retries": 0, "intent_": result.intent}


def route_decision(state: State) -> str:
    return state.get("intent_", "question")

In [15]:
# ✍️ TODO 2, 3, 4 — the retrieval loop's three nodes plus its decision function.


# ✍️ TODO 2, 3, 4 — the retrieval loop's three nodes plus its decision function.


def retrieve(state: State) -> dict:
    """TODO 2: run search_formatted on state['search_query'], return {'docs': ...}.
    Print a trace line: [retrieve] query='...'"""
    query = state["search_query"]
    print(f"[retrieve] query='{query}'")
    return {"docs": search_formatted(query)}


class Grade(BaseModel):
    relevant: bool = Field(
        description="True only if the documents contain enough information to answer "
        "the user's question. Identify the deciding passage before you decide."
    )


def grade(state: State) -> dict:
    """TODO 3a: llm.with_structured_output(Grade) on the question + docs.
    Return {'relevant_': result.relevant}; print [grade] relevant=..."""
    grader = llm.with_structured_output(Grade)
    result = grader.invoke(
        f"Question: {state['question']}\n\nDocuments:\n{state['docs']}"
    )
    print(f"[grade] relevant={result.relevant}")
    return {"relevant_": result.relevant}


def decide(state: State) -> str:
    """TODO 3b: return 'answer' if relevant_ OR retries >= MAX_RETRIES, else 'rewrite'.
    (Why must the retry cap live here, in code, rather than in a prompt?)"""
    if state["relevant_"] or state["retries"] >= MAX_RETRIES:
        return "answer"
    return "rewrite"


def rewrite(state: State) -> dict:
    """TODO 4: ask the LLM for ONE sharper handbook search query, given the question
    and the unhelpful docs. Return {'search_query': ..., 'retries': retries + 1}
    and print the new query."""
    result = llm.invoke(
        f"""Rewrite the following user question into ONE sharper search query for the handbook.

Question: {state['question']}

Unhelpful documents:
{state['docs']}

Return only the improved search query."""
    )

    if isinstance(result.content, list):
        new_query = " ".join(
            item.get("text", "") if isinstance(item, dict) else str(item)
            for item in result.content
        ).strip()
    else:
        new_query = result.content.strip()

    new_retries = state["retries"] + 1

    print(f"[rewrite] query='{new_query}'")

    return {
        "search_query": new_query,
        "retries": new_retries
    }

In [16]:
# Given: grounded generation, and the two refund nodes (human_gate lands in Part 5).
from langchain_core.messages import AIMessage
from langgraph.types import Command, interrupt


def answer(state: State) -> dict:
    prompt = (
        "Answer ONLY from these handbook excerpts, citing sources like [returns-policy]. "
        "If they don't contain the answer, say so honestly.\n\n"
        f"Excerpts:\n{state['docs']}\n\nQuestion: {state['question']}"
    )
    print("[answer]")
    return {"messages": [llm.invoke(state["messages"] + [("user", prompt)])]}


def human_gate(state: State) -> dict:
    """Part 5 — you will implement this. For now the graph simply auto-approves."""
    return {"approved_": True}


def process_refund(state: State) -> dict:
    if state.get("approved_"):
        msg = "Refund approved and queued — expect 5-10 business days. Ref #R-2210."
    else:
        msg = "A human reviewed this request and it was not approved. You can reply to escalate."
    print(f"[process_refund] approved={state.get('approved_')}")
    return {"messages": [AIMessage(msg)]}

In [17]:
# ✍️ TODO 5 — wire the graph so it matches the diagram exactly, then compile it.
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph


def build_graph():
    g = StateGraph(State)
    # TODO a) add all seven nodes: router, retrieve, grade, rewrite, answer,
    #         human_gate, process_refund
    g.add_node("router", router)
    g.add_node("retrieve", retrieve)
    g.add_node("grade", grade)
    g.add_node("rewrite", rewrite)
    g.add_node("answer", answer)
    g.add_node("human_gate", human_gate)
    g.add_node("process_refund", process_refund)

    # TODO b) g.add_edge(START, "router")
    g.add_edge(START, "router")

    # TODO c) g.add_conditional_edges("router", route_decision,
    #             {"question": "retrieve", "refund": "human_gate"})
    g.add_conditional_edges(
        "router",
        route_decision,
        {"question": "retrieve", "refund": "human_gate"}
    )

    # TODO d) retrieve -> grade;  grade -conditional(decide)-> answer | rewrite
    g.add_edge("retrieve", "grade")
    g.add_conditional_edges(
        "grade",
        decide,
        {"answer": "answer", "rewrite": "rewrite"}
    )

    # TODO e) rewrite -> retrieve      ← the self-correction loop, one edge
    g.add_edge("rewrite", "retrieve")

    # TODO f) answer -> END;  human_gate -> process_refund -> END
    g.add_edge("answer", END)
    g.add_edge("human_gate", "process_refund")
    g.add_edge("process_refund", END)

    # The checkpointer gives per-thread memory AND makes Part 5's interrupt possible.
    return g.compile(checkpointer=InMemorySaver())


app = build_graph()
print(app.get_graph().draw_mermaid())    # paste into https://mermaid.live — is it the diagram?

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	router(router)
	retrieve(retrieve)
	grade(grade)
	rewrite(rewrite)
	answer(answer)
	human_gate(human_gate)
	process_refund(process_refund)
	__end__([<p>__end__</p>]):::last
	__start__ --> router;
	grade -.-> answer;
	grade -.-> rewrite;
	human_gate --> process_refund;
	retrieve --> grade;
	rewrite --> retrieve;
	router -. &nbsp;refund&nbsp; .-> human_gate;
	router -. &nbsp;question&nbsp; .-> retrieve;
	answer --> __end__;
	process_refund --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [18]:
# Test the happy path and the self-correcting path.
def run(question: str, thread_id: str = "demo") -> str:
    config = {"configurable": {"thread_id": thread_id}}
    state = app.invoke({"messages": [("user", question)], "question": question}, config)
    print("\nFINAL:", state["messages"][-1].content, "\n")
    return state["messages"][-1].content


run("What's the restocking fee on opened electronics?", thread_id="t1")
run("whats the deal with wet tents lol", thread_id="t2")     # vague → watch rewrite fire

D:\SE3090_Lab06_Agentic_AI_Part_2\SE3090_Lab06_Agentic_AI_Part_2\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[router] intent = question
[retrieve] query='What's the restocking fee on opened electronics?'


D:\SE3090_Lab06_Agentic_AI_Part_2\SE3090_Lab06_Agentic_AI_Part_2\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[grade] relevant=True
[answer]


D:\SE3090_Lab06_Agentic_AI_Part_2\SE3090_Lab06_Agentic_AI_Part_2\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


...

D:\SE3090_Lab06_Agentic_AI_Part_2\SE3090_Lab06_Agentic_AI_Part_2\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[grade] relevant=False
[answer]


D:\SE3090_Lab06_Agentic_AI_Part_2\SE3090_Lab06_Agentic_AI_Part_2\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



FINAL: [{'type': 'text', 'text': 'The provided handbook excerpts do not contain information about wet tents.', 'extras': {'signature': 'El4KXAERTTIPZxXJ0wqu8ycj22OgAc9X164andDid4nQmtlpcQ3fajcoXe8DocTbHk1CUrgdlTmng1oh1TedbeJJb4k0qK4ThIFh7BoUyUqdv7S4Lih+XH03fEN9vd59'}}] 



[{'type': 'text',
  'text': 'The provided handbook excerpts do not contain information about wet tents.',
  'extras': {'signature': 'El4KXAERTTIPZxXJ0wqu8ycj22OgAc9X164andDid4nQmtlpcQ3fajcoXe8DocTbHk1CUrgdlTmng1oh1TedbeJJb4k0qK4ThIFh7BoUyUqdv7S4Lih+XH03fEN9vd59'}}]

✅ **CHECKPOINT 3** — Show a run where `[grade] relevant = False` fired `[rewrite]`, and the second retrieval succeeded. Show your mermaid diagram next to the one above.
*Demonstrator question: which single line of your code is the self-correction loop?*

## Part 4 — Memory via checkpointing (1:20–1:33)

You already compiled with `checkpointer=InMemorySaver()`, and every call passes `thread_id`. That is the whole feature: **state is saved after every node, keyed by thread**. Memory is not something the model has; it is something your runtime keeps.

In [20]:
# Same thread → the follow-up has context.
run("What's the return window for tents?", thread_id="alice")
run("And how long do refunds take to arrive?", thread_id="alice")   # note: no re-asking

D:\SE3090_Lab06_Agentic_AI_Part_2\SE3090_Lab06_Agentic_AI_Part_2\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[router] intent = question
[retrieve] query='What's the return window for tents?'


D:\SE3090_Lab06_Agentic_AI_Part_2\SE3090_Lab06_Agentic_AI_Part_2\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[grade] relevant=True
[answer]


D:\SE3090_Lab06_Agentic_AI_Part_2\SE3090_Lab06_Agentic_AI_Part_2\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


...

D:\SE3090_Lab06_Agentic_AI_Part_2\SE3090_Lab06_Agentic_AI_Part_2\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[grade] relevant=True
[answer]


D:\SE3090_Lab06_Agentic_AI_Part_2\SE3090_Lab06_Agentic_AI_Part_2\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



FINAL: [{'type': 'text', 'text': 'Refunds typically arrive within **5–10 business days** of receiving the item [returns-policy].', 'extras': {'signature': 'El4KXAERTTIPw8gtNkub4TizU2e1hETTO7GVnxW3k6j69lPCeZ46Hb+JJXImIQimlltX6+NkPDsvlMs8BUtQMO6du7nYUguMBvfg//oNxtmzZ5DIojRFG0kDGOZfPqEk'}}] 



[{'type': 'text',
  'text': 'Refunds typically arrive within **5–10 business days** of receiving the item [returns-policy].',
  'extras': {'signature': 'El4KXAERTTIPw8gtNkub4TizU2e1hETTO7GVnxW3k6j69lPCeZ46Hb+JJXImIQimlltX6+NkPDsvlMs8BUtQMO6du7nYUguMBvfg//oNxtmzZ5DIojRFG0kDGOZfPqEk'}}]

In [21]:
# ✍️ EXERCISE 4.1 — same follow-up, FRESH thread. What happens, and why?
run("And how long do refunds take to arrive?", thread_id="stranger")

D:\SE3090_Lab06_Agentic_AI_Part_2\SE3090_Lab06_Agentic_AI_Part_2\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[router] intent = question
[retrieve] query='And how long do refunds take to arrive?'


D:\SE3090_Lab06_Agentic_AI_Part_2\SE3090_Lab06_Agentic_AI_Part_2\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[grade] relevant=True
[answer]


D:\SE3090_Lab06_Agentic_AI_Part_2\SE3090_Lab06_Agentic_AI_Part_2\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



FINAL: [{'type': 'text', 'text': 'Refunds typically arrive within **5–10 business days** of receiving the item [returns-policy].', 'extras': {'signature': 'El4KXAERTTIPt1kx+gTuipGOep82/8I0rozTvkhIDO94aFd3F4eBglLtlLouCcKKs8Bpdsig6E5ViU51Aa1kfl3yuufmmvyUh23fnudPdzUNoYp/WocRCUu1NP7I14qo'}}] 



[{'type': 'text',
  'text': 'Refunds typically arrive within **5–10 business days** of receiving the item [returns-policy].',
  'extras': {'signature': 'El4KXAERTTIPt1kx+gTuipGOep82/8I0rozTvkhIDO94aFd3F4eBglLtlLouCcKKs8Bpdsig6E5ViU51Aa1kfl3yuufmmvyUh23fnudPdzUNoYp/WocRCUu1NP7I14qo'}}]

In [22]:
# Memory, made inspectable: what the checkpointer holds for a thread.
snapshot = app.get_state({"configurable": {"thread_id": "alice"}})
print("stored fields :", sorted(snapshot.values))
print("messages      :", len(snapshot.values["messages"]))
for m in snapshot.values["messages"]:
    print(f"  {type(m).__name__:<13} {str(m.content)[:80]}")

stored fields : ['docs', 'intent_', 'messages', 'question', 'relevant_', 'retries', 'search_query']
messages      : 7
  HumanMessage  What's the return window for tents?
  AIMessage     [{'type': 'text', 'text': 'Tents and shelters are returnable within **14 days of
  HumanMessage  And how long do refunds take to arrive?
  HumanMessage  What's the return window for tents?
  AIMessage     [{'type': 'text', 'text': 'Tents and shelters are returnable within **14 days of
  HumanMessage  And how long do refunds take to arrive?
  AIMessage     [{'type': 'text', 'text': 'Refunds typically arrive within **5–10 business days*


✅ **CHECKPOINT 4** — Show the follow-up working on `alice` and failing on `stranger`, plus the thread snapshot.

## Part 5 — Human-in-the-loop refunds (1:33–1:45)

Money is the classic place to stop and ask a person. LangGraph's `interrupt()` does something stronger than "wait for input": it **checkpoints the entire graph and ends the run**. The process can exit; hours can pass; the resume carries only the decision.

In [23]:
# ✍️ TODO 6 — implement the gate, then rebuild the graph so it takes effect.
def human_gate(state: State) -> dict:
    # TODO: decision = interrupt({"ask": f"Refund requested: {state['question']!r}. Approve?"})
    #       return {"approved_": str(decision).lower().startswith("approve")}
    
    decision = interrupt({
        "ask": f"Refund requested: {state['question']!r}. Approve?"
    })
    
    return {
        "approved_": str(decision).lower().startswith("approve")
    }


app = build_graph()      # rebuild so the graph uses your new human_gate

In [24]:
# Run a refund request: the graph will pause instead of finishing.
config = {"configurable": {"thread_id": "refund-1"}}
question = "I want a refund for my NightHawk headlamp, order #4417"

state = app.invoke({"messages": [("user", question)], "question": question}, config)

if state.get("__interrupt__"):
    print("PAUSED — the graph is frozen in the checkpointer.")
    print("payload:", state["__interrupt__"][0].value)
else:
    print("no interrupt fired — did you rebuild the graph after TODO 6?")

D:\SE3090_Lab06_Agentic_AI_Part_2\SE3090_Lab06_Agentic_AI_Part_2\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[router] intent = refund
PAUSED — the graph is frozen in the checkpointer.
payload: {'ask': "Refund requested: 'I want a refund for my NightHawk headlamp, order #4417'. Approve?"}


In [25]:
# The human decides. Nothing about the original request is resent.
state = app.invoke(Command(resume="approve"), config)
print("FINAL:", state["messages"][-1].content)

[process_refund] approved=True
FINAL: Refund approved and queued — expect 5-10 business days. Ref #R-2210.


In [26]:
# ✍️ EXERCISE 5.1 — now deny one, on a NEW thread. Confirm process_refund still runs
# but takes the other branch, and that the customer gets an honest message.
config_deny = {"configurable": {"thread_id": "refund-2"}}
q2 = "Refund my clearance jacket please, order #9001"

state = app.invoke({"messages": [("user", q2)], "question": q2}, config_deny)
print("payload:", state["__interrupt__"][0].value)
state = app.invoke(Command(resume="deny"), config_deny)
print("FINAL:", state["messages"][-1].content)

D:\SE3090_Lab06_Agentic_AI_Part_2\SE3090_Lab06_Agentic_AI_Part_2\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[router] intent = refund
payload: {'ask': "Refund requested: 'Refund my clearance jacket please, order #9001'. Approve?"}
[process_refund] approved=False
FINAL: A human reviewed this request and it was not approved. You can reply to escalate.


**Where did the state live between the pause and your decision?** In the checkpointer — which is why Part 5 is impossible without Part 4. Compile without a checkpointer and `interrupt()` raises. (Try it if you have a minute; the error message is the lesson.)

✅ **CHECKPOINT 5** — Show one approved and one denied refund flow, including the interrupt payloads.

## Part 6 — Serve the whole workflow behind FastAPI (1:45–1:55)

`api/main.py` in this folder is everything you just built, packaged as a service. The interesting part is that **the human gate survives the packaging**:

| Endpoint | What it does |
|---|---|
| `GET /health` | Service + index status (no model call) |
| `POST /ingest` | Rebuild the index from `./data` |
| `GET /search?q=…` | Raw hybrid search — no agent involved |
| `POST /ask` | Run the graph; returns an answer **or** `awaiting_approval` + payload |
| `POST /resume` | A human's decision; the graph continues from where it froze |
| `GET /threads/{id}` | What the checkpointer remembers for that thread |

Two HTTP calls, one workflow, a human in between. Read the file, then run it.

In [27]:
# Start the API in a background thread (or run `uvicorn api.main:app --reload` in a terminal)
import socket
import sys
import threading
import time

import requests
import uvicorn

sys.path.insert(0, str(LAB_DIR))
from api.main import app as api_app

with socket.socket() as s:
    s.bind(("127.0.0.1", 0))
    PORT = s.getsockname()[1]

BASE = f"http://127.0.0.1:{PORT}"

server = uvicorn.Server(
    uvicorn.Config(
        api_app,
        host="127.0.0.1",
        port=PORT,
        log_level="warning"
    )
)

threading.Thread(target=server.run, daemon=True).start()

# Wait for the API to become available
api_ready = False

for _ in range(40):
    try:
        response = requests.get(
            f"{BASE}/health",
            timeout=3
        )

        if response.status_code == 200:
            api_ready = True
            break

    except (
        requests.exceptions.ConnectionError,
        requests.exceptions.ReadTimeout
    ):
        time.sleep(0.5)

if api_ready:
    print("API up  ->", BASE)
    print("Swagger ->", f"{BASE}/docs")
    print(requests.get(f"{BASE}/health", timeout=5).json())
else:
    print("API did not become ready.")
    print("Try running this in a terminal:")
    print(f"uvicorn api.main:app --reload")

API up  -> http://127.0.0.1:3240
Swagger -> http://127.0.0.1:3240/docs
{'status': 'ok', 'provider': 'Google AI Studio (Gemini API, free tier)', 'model': 'gemini-3.5-flash-lite', 'embedding_model': 'gemini-embedding-001', 'chroma_dir': 'D:\\SE3090_Lab06_Agentic_AI_Part_2\\SE3090_Lab06_Agentic_AI_Part_2\\chroma_db', 'indexed_chunks': 8}


In [28]:
# Retrieval over HTTP (the index you built in Part 1 — same folder, same collection)
for hit in requests.get(f"{BASE}/search", params={"q": "tent return window", "k": 2}).json():
    print(f"[{hit['source']}] {hit['text'][:100]}…")

[returns-policy] # Contoso Outdoor Gear — Returns & Refunds Policy (2026 edition)

## Standard return window

Most it…
[returns-policy] ## Refund processing

Refunds are issued to the **original payment method** and typically arrive wit…


In [29]:
# A two-turn conversation on one thread: memory works across separate HTTP requests
r1 = requests.post(f"{BASE}/ask", json={"question": "What's the return window for tents?",
                                        "thread_id": "web-user-7"}, timeout=180).json()
print("turn 1 :", r1["answer"], "\n   nodes:", r1["nodes"])

r2 = requests.post(f"{BASE}/ask", json={"question": "And how long do refunds take to arrive?",
                                        "thread_id": "web-user-7"}, timeout=180).json()
print("turn 2 :", r2["answer"], "\n   nodes:", r2["nodes"])

D:\SE3090_Lab06_Agentic_AI_Part_2\SE3090_Lab06_Agentic_AI_Part_2\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
D:\SE3090_Lab06_Agentic_AI_Part_2\SE3090_Lab06_Agentic_AI_Part_2\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
D:\SE3090_Lab06_Agentic_AI_Part_2\SE3090_Lab06_Agentic_AI_Part_2\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


turn 1 : Tents and shelters are returnable within **14 days of delivery** [returns-policy]. 
   nodes: ['router', 'retrieve', 'grade', 'answer']


D:\SE3090_Lab06_Agentic_AI_Part_2\SE3090_Lab06_Agentic_AI_Part_2\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
D:\SE3090_Lab06_Agentic_AI_Part_2\SE3090_Lab06_Agentic_AI_Part_2\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
D:\SE3090_Lab06_Agentic_AI_Part_2\SE3090_Lab06_Agentic_AI_Part_2\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


turn 2 : Refunds typically arrive within **5–10 business days** of receiving the item [returns-policy]. 
   nodes: ['router', 'retrieve', 'grade', 'answer']


In [30]:
# The human gate, over HTTP: /ask pauses, a person decides, /resume finishes.
paused = requests.post(f"{BASE}/ask", json={"question": "I want a refund for order #4417",
                                            "thread_id": "web-user-9"}, timeout=180).json()
print("status  :", paused["status"])
print("payload :", paused["interrupt"])

done = requests.post(f"{BASE}/resume", json={"thread_id": "web-user-9",
                                             "decision": "approve"}, timeout=180).json()
print("resumed :", done["answer"])
print("nodes   :", done["nodes"])

D:\SE3090_Lab06_Agentic_AI_Part_2\SE3090_Lab06_Agentic_AI_Part_2\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


status  : awaiting_approval
payload : {'ask': "Refund requested: 'I want a refund for order #4417'. Approve?"}
resumed : Refund approved and queued — expect 5-10 business days. Ref #R-2210.
nodes   : ['human_gate', 'process_refund']


In [31]:
# ✍️ EXERCISE 6.1 — inspect the server's memory of a thread, then answer in a comment:
#   a) A different customer hits /ask with thread_id="web-user-7". What do they see,
#      and why is thread_id therefore a security-relevant identifier?
#   b) The server restarts while a refund is awaiting approval. What is lost, and which
#      one-line change (see stretch goal 4) fixes it?
print(requests.get(f"{BASE}/threads/web-user-7").json())

# a) The different customer sees the conversation/state stored under "web-user-7".
#    Since the thread_id is used to retrieve the conversation, reusing or guessing
#    another customer's thread_id could expose their messages and state. Therefore,
#    thread_id is a security-relevant identifier and must be protected/authorized.
#
# b) With InMemorySaver(), the thread state and pending human approval are lost when
#    the server restarts because memory is only stored in RAM.
#    The fix is to use a persistent checkpointer/database instead of InMemorySaver(),
#    e.g. replace InMemorySaver() with SqliteSaver backed by a SQLite database.

{'thread_id': 'web-user-7', 'question': 'And how long do refunds take to arrive?', 'retries': 0, 'messages': [{'type': 'HumanMessage', 'content': "What's the return window for tents?"}, {'type': 'AIMessage', 'content': "[{'type': 'text', 'text': 'Tents and shelters are returnable within **14 days of delivery** [returns-policy].', 'extras': {'signature': 'El4KXAERTTIPO6FE08xc8bPUs6fzjHtANUvvpQdvORLYmOHigjlXofZBpQsLbv/g3gesuW7YC52rfo87bl7blJZD1j3j/SnlAuD+Mm2V2wECVY3va2/9TkF1yxAHF7+y'}}]"}, {'type': 'HumanMessage', 'content': 'And how long do refunds take to arrive?'}, {'type': 'AIMessage', 'content': "[{'type': 'text', 'text': 'Refunds typically arrive within **5–10 business days** of receiving the item [returns-policy].', 'extras': {'signature': 'El4KXAERTTIPcnxDhJK0+ThYZhXFMmZRR5X752GKx87mXoXpwqG/Hi4ZHii7O61GhMKuW8mznnfDSPc18vgntJVo7ioXRkFzL82VSR+Dx6SpGkAxnjSvBhv5g1sQ/2UY'}}]"}]}


D:\SE3090_Lab06_Agentic_AI_Part_2\SE3090_Lab06_Agentic_AI_Part_2\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
D:\SE3090_Lab06_Agentic_AI_Part_2\SE3090_Lab06_Agentic_AI_Part_2\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "D:\SE3090_Lab06_Agentic_AI_Part_2\SE3090_Lab06_Agentic_AI_Part_2\.venv\Lib\site-packages\uvicorn\protocols\http\httptools_impl.py", line 422, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        self.scope, self.receive, self.se

✅ **CHECKPOINT 6** — Show the two-turn memory over HTTP and one full pause→resume refund cycle, plus your answers to Exercise 6.1.

## Wrap-up (1:55–2:00)

Today's agent is **grounded** (it answers from an index you built and cites it), **self-correcting** (grade → rewrite → retry), **stateful** (checkpointed threads), **supervised** (a human gate on money), and **deployable** (a service where the pause survives the request boundary). That is a production-shaped agent. What remains for Week 7: scale-out (multiple agents), proof (evaluation), and armour (security).

**Deliverables:** `lab6_evidence.txt` with the six checkpoints, plus this notebook with Parts 1, 3 and 5 completed.

### Stretch goals

1. **Summarization node** — after 8+ messages on a thread, compress older turns into a summary message. Verify that token usage drops on turn 9.
2. **Citation checker** — a node after `answer` that verifies every `[source]` cited actually appears in `docs`, and flags the answer if not. That is a groundedness guard, and it previews Week 7's evaluation.
3. **Parent-document retrieval** — re-ingest with small chunks carrying a `parent` metadata field holding the full section; retrieve small, feed the model big.
4. **Durable state** — swap `InMemorySaver` for `SqliteSaver` (`langgraph-checkpoint-sqlite`, already installed). Start a refund via the API, kill the server, restart it, then `POST /resume`. Durability, demonstrated.
5. **Streaming answers** — add `POST /ask/stream` to `api/main.py` that yields node updates as server-sent events.

### Troubleshooting

| Symptom | Likely cause |
|---|---|
| "Index is empty" / no search results | Ingest cell not run, or `chroma_db/` deleted — re-run Part 1 |
| History disappears between nodes | Missing `add_messages` reducer in `State` (TODO 1) |
| `interrupt` raises about a checkpointer | You compiled without `checkpointer=` |
| Interrupt never fires | You rebuilt nothing after TODO 6 — re-run `app = build_graph()` |
| Grader always says relevant | Grading prompt too soft — make it identify the deciding passage first |
| Embedding call fails | Check `EMBEDDING_MODEL` is `gemini-embedding-001` and the key is valid |
| `429` / `ResourceExhausted` during Part 3 | Free-tier per-minute cap — wait 60 s; the graph is retry-friendly |
| API cell hangs | An old server thread is still bound — restart the kernel and rerun |